# Dataset loader
We will use this script to convert image dataset to pkl file format

## KITTI dataset

In [1]:
import os
import pickle
import numpy as np
from PIL import Image
from glob import glob
import matplotlib.pyplot as plt
from tqdm import tqdm

In [4]:
def pickler(DATA_PATH):
    images = sorted(glob(DATA_PATH))
    terrain_patches = []

    for img in tqdm(images):
        img = Image.open(img).convert('L')
        img = img.resize((40, 40))
        terrain_patches.append(np.array(img))

    terrain_patches = np.array(terrain_patches)
    print(len(terrain_patches))

    with open("./vertiencoder/data/train/data_train.pkl", "wb") as f:
        pickle.dump(terrain_patches, f)

In [5]:
DIR_PATH = os.path.abspath('.')
# IMAG_PATH = os.path.join(DIR_PATH, 'dataset', 'RUGD', 'RUGD_frames-with-annotations', 'creek', '*.png')
IMAG_PATH = os.path.join(DIR_PATH, 'dataset', 'test_data1', 'cam', 'rgb', '*.png')

pickler(IMAG_PATH)

100%|██████████| 341/341 [00:02<00:00, 134.27it/s]

341


## Davis Hall dataset

In [21]:
import os
import json
import pickle
import numpy as np
from glob import glob
from PIL import Image
from tqdm import tqdm

In [22]:
# Path to dataset folder
DATASET_PATH = "./dataset/test_data1"

# Load Camera Parameters
with open(os.path.join(DATASET_PATH, "cam_param.json"), "r") as f:
    cam_params = json.load(f)

# Load LiDAR Parameters
with open(os.path.join(DATASET_PATH, "lidar_param.json"), "r") as f:
    lidar_params = json.load(f)

# Load IMU Parameters
with open(os.path.join(DATASET_PATH, "imu_param.json"), "r") as f:
    imu_params = json.load(f)

In [23]:
# Load Motion Data
pose_data = np.loadtxt(os.path.join(DATASET_PATH, "pose.txt"))  # (x, y, z, roll, pitch, yaw)
vel_data = np.loadtxt(os.path.join(DATASET_PATH, "vel.txt"))  # (v, ω)
wheel_speed = np.loadtxt(os.path.join(DATASET_PATH, "wheel_speed.txt"))  # Motor speeds
timestamps = np.loadtxt(os.path.join(DATASET_PATH, "timestamp.txt"))  # Time

In [24]:
def load_footprint_images():
    # Load RGB Images (Convert to 40x40 Grayscale)
    image_paths = sorted(glob(os.path.join(DATASET_PATH, "cam/rgb/*.png")))  # Load RGB images
    footprint_images = []
    for img_path in tqdm(image_paths):
        img = Image.open(img_path).convert("L")  # Convert to grayscale
        img = img.resize((40, 40))  # Resize to 40x40
        footprint_images.append(np.array(img))

    footprint_images = np.array(footprint_images)
    return footprint_images


def load_footprint_paths():
    image_paths = os.listdir('./dataset/test_data1/footprint')
    footprint_paths = [ f'./footprint/{path}' for path in image_paths ]
    return footprint_paths


# footprint_images = load_footprint_images()
footprints = load_footprint_paths()

In [25]:
def create_patches():
    # Load RGB Images (Convert to 40x40 Grayscale)
    image_paths = sorted(glob(os.path.join(DATASET_PATH, "cam/rgb/*.png")))  # Load RGB images
    # footprint_images = []
    for img_path in tqdm(image_paths):
        img = Image.open(img_path).convert("L")  # Convert to grayscale
        img = img.resize((40, 40))  # Resize to 40x40
        # footprint_images.append(np.array(img))
        np.save()

    footprint_images = np.array(footprint_images)
    return footprint_images

In [26]:
# Define paths
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm

DATASET_DIR = Path("./dataset/test_data1")
FOOTPRINT_DIR = DATASET_DIR / "footprint"

# Create footprint directory if it doesn't exist
FOOTPRINT_DIR.mkdir(exist_ok=True)

# Process depth images
depth_path = DATASET_DIR / "cam" / "depth"
depth_files = sorted(depth_path.glob("*.png"))  # Change to actual depth image format

for idx, depth_file in tqdm(enumerate(depth_files), total=len(depth_files)):
    # Load depth image
    depth_img = cv2.imread(str(depth_file), cv2.IMREAD_UNCHANGED)  # Load as grayscale

    # Normalize height values (example: scale between -1 and 1)
    height_map = (depth_img - np.min(depth_img)) / (np.max(depth_img) - np.min(depth_img)) * 2 - 1

    # Save as .npy file
    npy_filename = FOOTPRINT_DIR / f"mocap_{idx}.npy"
    np.save(npy_filename, height_map)

    # print(f"Saved footprint: {npy_filename}")


  0%|          | 0/341 [00:00<?, ?it/s]

100%|██████████| 341/341 [00:05<00:00, 62.33it/s]


In [27]:
# Compute Time Differences
dt_values = np.diff(timestamps, prepend=timestamps[0])

# Compute Pose Differences
pose_diff = np.diff(pose_data, axis=0, prepend=pose_data[0:1, :])

# Create Dataset Dictionary
dataset = {
    "bag_name": ["test_data1"],
    "data": []
}

# Structure Data for Training
num_samples = min(len(footprints), len(vel_data), len(pose_data))  # Ensure matching sizes

sample = {
    "cmd_vel": vel_data,  # (v, ω)
    "footprint": footprints,  # 40x40 grayscale image
    "pose": pose_data,  # (x, y, z, roll, pitch, yaw)
    "motor_speed": wheel_speed,  # If missing, use zeros
    "dt": np.array(dt_values),  # Time difference
    "pose_diff": pose_diff,  # Pose changes
    "time": np.array(timestamps)  # Timestamp
}

dataset["data"].append(sample)

In [28]:
# Save Processed Data
with open("./vertiencoder/data/train/data_total.pkl", "wb") as f:
    pickle.dump(dataset, f)

print("✅ Dataset saved as data_total.pkl with", num_samples, "samples.")


✅ Dataset saved as data_total.pkl with 341 samples.


### Creating training split

In [29]:
train_dataset = {
    "bag_name": dataset["bag_name"],
    "data":  [{}]
}

TRAIN_SPLIT = int(0.8 * num_samples)

for key in dataset['data'][0].keys():
    train_dataset['data'][0][key] = dataset['data'][0][key][:TRAIN_SPLIT]

In [30]:
len(train_dataset['data'][0]['cmd_vel'])

272

In [31]:
# Save Processed Data
with open("./vertiencoder/data/train/data_train.pkl", "wb") as f:
    pickle.dump(dataset, f)

In [32]:
# from shutil import copy

# SOURCE_PATH = './dataset/test_data1/footprint/'
# footprints = os.listdir(SOURCE_PATH)
# DEST_PATH = './vertiencoder/data/train/footprint/'

# TRAIN_SPLIT = int(0.8 * len(footprints))
# train_footprints = footprints[:TRAIN_SPLIT]
# val_footprints = footprints[TRAIN_SPLIT:]

# for patch in train_footprints:
#     src_path = os.path.join(SOURCE_PATH, patch)
#     copy(src_path, DEST_PATH)

In [33]:
# DEST_PATH = './vertiencoder/data/val/footprint/'

# for patch in val_footprints:
#     src_path = os.path.join(SOURCE_PATH, patch)
#     copy(src_path, DEST_PATH)

### Creating validation dataset

In [34]:
valid_dataset = {
    "bag_name": dataset["bag_name"],
    "data":  [{}]
}

TRAIN_SPLIT = int(0.8 * num_samples)

for key in dataset['data'][0].keys():
    valid_dataset['data'][0][key] = dataset['data'][0][key][TRAIN_SPLIT:]

In [35]:
len(valid_dataset['data'][0]['cmd_vel'])

69

In [36]:
with open('./vertiencoder/data/val/data_val.pkl', 'wb') as f:
    pickle.dump(valid_dataset, f)

### Running stats script

In [116]:
! python ./vertiencoder/utils/stats.py

name: train
cmd_vel_mean: tensor([-0.0091, -0.0715,  0.0006])
cmd_vel_var: tensor([4.4420e-01, 5.7062e-01, 3.2379e-04])
cmd_vel_std: tensor([0.6665, 0.7554, 0.0180])
cmd_vel_max: tensor([1.0406, 1.1557, 0.0960])
cmd_vel_min: tensor([-1.1616, -1.2199, -0.0610])
footprint_mean: -0.6748576164245605
footprint_var: 0.3512652814388275
footprint_std: 0.5926763415336609
footprint_max: 1.0
footprint_min: -1.0
pose_mean: tensor([ 8.3858e-02, -2.1267e+00,  2.9450e-02, -1.8435e-03, -1.4351e-03,
        -1.8636e-01,  6.9233e-01])
pose_var: tensor([6.7629e+00, 8.4924e+00, 1.3451e-03, 1.4474e-05, 6.5411e-05, 3.8486e-01,
        1.0242e-01])
pose_std: tensor([2.6006, 2.9142, 0.0367, 0.0038, 0.0081, 0.6204, 0.3200])
pose_max: tensor([4.6685, 3.0087, 0.1024, 0.0127, 0.0157, 0.6575, 1.0000])
pose_min: tensor([-5.1115, -7.3916, -0.0389, -0.0181, -0.0222, -0.9961,  0.0861])
motor_speed_mean: tensor([0.9376, 0.9253, 0.9376, 0.9253])
motor_speed_var: tensor([0.0703, 0.0694, 0.0703, 0.0694])
motor_speed_std: 

### Opening the pickle file

In [89]:
with open('./vertiencoder/data/train/data_train.pkl', 'rb') as f:
    global content
    content = pickle.load(f)
    print(content.keys())

dict_keys(['bag_name', 'data'])


In [90]:
content['data']

[{'cmd_vel': array([[-0.01793677, -0.178919  ,  0.00371403],
         [-0.01793677, -0.178919  ,  0.00371403],
         [-0.01000372, -0.22223623,  0.00637325],
         ...,
         [ 0.37625283, -0.25593996, -0.01059414],
         [ 0.29400232, -0.20700024, -0.00284659],
         [ 0.29400232, -0.20700024, -0.00284659]]),
  'footprint': ['./footprint/mocap_0.npy',
   './footprint/mocap_1.npy',
   './footprint/mocap_10.npy',
   './footprint/mocap_100.npy',
   './footprint/mocap_101.npy',
   './footprint/mocap_102.npy',
   './footprint/mocap_103.npy',
   './footprint/mocap_104.npy',
   './footprint/mocap_105.npy',
   './footprint/mocap_106.npy',
   './footprint/mocap_107.npy',
   './footprint/mocap_108.npy',
   './footprint/mocap_109.npy',
   './footprint/mocap_11.npy',
   './footprint/mocap_110.npy',
   './footprint/mocap_111.npy',
   './footprint/mocap_112.npy',
   './footprint/mocap_113.npy',
   './footprint/mocap_114.npy',
   './footprint/mocap_115.npy',
   './footprint/mocap_116.

### Sample data

In [2]:
import pickle
with open('./dataset/test_data1/data_train.pickle', 'rb') as f:
    content2 = pickle.load(f)

content2.keys()

dict_keys(['bag_name', 'data', 'total_records'])

In [4]:
content2['data'][0]['footprint']

['footprint/P05_0.npy',
 'footprint/P05_1.npy',
 'footprint/P05_2.npy',
 'footprint/P05_3.npy',
 'footprint/P05_4.npy',
 'footprint/P05_5.npy',
 'footprint/P05_6.npy',
 'footprint/P05_7.npy',
 'footprint/P05_8.npy',
 'footprint/P05_9.npy',
 'footprint/P05_10.npy',
 'footprint/P05_11.npy',
 'footprint/P05_12.npy',
 'footprint/P05_13.npy',
 'footprint/P05_14.npy',
 'footprint/P05_15.npy',
 'footprint/P05_16.npy',
 'footprint/P05_17.npy',
 'footprint/P05_18.npy',
 'footprint/P05_19.npy',
 'footprint/P05_20.npy',
 'footprint/P05_21.npy',
 'footprint/P05_22.npy',
 'footprint/P05_23.npy',
 'footprint/P05_24.npy',
 'footprint/P05_25.npy',
 'footprint/P05_26.npy',
 'footprint/P05_27.npy',
 'footprint/P05_28.npy',
 'footprint/P05_29.npy',
 'footprint/P05_30.npy',
 'footprint/P05_31.npy',
 'footprint/P05_32.npy',
 'footprint/P05_33.npy',
 'footprint/P05_34.npy',
 'footprint/P05_35.npy',
 'footprint/P05_36.npy',
 'footprint/P05_37.npy',
 'footprint/P05_38.npy',
 'footprint/P05_39.npy',
 'footprin

### Exploring given dataset

In [39]:
arr1 = np.load('./dataset/test_data1/cam/true_depth/000000.npz')
arr2 = np.load('./dataset/test_data1/lidar/000000.npz')

In [42]:
print(arr1.keys())
print(arr2.keys())

KeysView(NpzFile './dataset/test_data1/cam/true_depth/000000.npz' with keys: depth)
KeysView(NpzFile './dataset/test_data1/lidar/000000.npz' with keys: points)


In [45]:
print(arr1['depth'].shape)
print(arr2['points'].shape)

(480, 640)
(3305, 3)


In [ ]:
arr2['points'].

array([[-3.8056152,  2.9050524,  1.43451  ],
       [-3.9716713,  2.989953 ,  1.3425138],
       [-4.169726 ,  3.0768416,  1.2585225],
       ...,
       [-5.1900077,  1.6947064,  1.9961307],
       [-5.3940077,  1.6217066,  1.8881269],
       [-5.6150084,  1.5327067,  1.7931231]], dtype=float32)